# BBDM(LBBDM-f4) 리파이너 실험 — 학습·샘플·평가 (직접 실행)

제안서 마지막 후보 **BBDM**(Brownian Bridge Diffusion, 잠재공간 f4). 프레임워크를 **노트북 셀에서 직접 호출** → loss 스트리밍 관찰.

- 커널 `trellis` → Restart Kernel → Run All
- 잠재공간(64) 학습이라 가벼움: **1 epoch ≈ 47초**. `MAX_EPOCH` 로 조절 (기본 40 ≈ 30분; 제대로면 100 ≈ 1.5h)
- 자동 저장(2 epoch마다). 커널 죽으면 셀2 건너뛰고 셀3(샘플)부터 — 디스크 체크포인트 로드
- ⚠️ BBDM은 **paired(정렬쌍)** 방식 — CUT/UNSB(unpaired)와 카테고리 다름. 참고로 비교.
- 비교 기준(test 60 동일표본): plain CUT / CycleGAN / UNSB (이전 저장 pngs)
- ⚠️ **육안 검증 필수**(셀 5) — 지표는 여러 번 속았음([[교훈]])


## 0. 환경 + BBDM 실행 헬퍼

In [ ]:
import os, sys, glob, time, argparse, yaml, random
import numpy as np, torch
from PIL import Image
import matplotlib.pyplot as plt
ROOT = os.path.expanduser("~/erang_sr")
BBDM = os.path.join(ROOT, "refiners/bbdm_src")
os.chdir(BBDM); sys.path.insert(0, BBDM)          # 로컬 패키지(datasets/models/runners) 우선
DEV = "cuda"
CFG = "configs/lbbdm_hat_f4.yaml"
MAX_EPOCH = 40      # ★ 학습 epoch (기본 40≈30분, 제대로면 100)

from utils import dict2namespace, get_runner
def set_seed(s=1234):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True

def run_bbdm(train, resume_model=None, max_epoch=None):
    args = argparse.Namespace(config=CFG, seed=1234, result_path="results",
        train=train, sample_to_eval=(not train), sample_at_start=False, save_top=True,
        gpu_ids="0", port="12355", resume_model=resume_model, resume_optim=None,
        max_epoch=max_epoch, max_steps=None)
    with open(CFG) as f: dc = yaml.load(f, Loader=yaml.FullLoader)
    config = dict2namespace(dc); config.args = args
    if resume_model is not None: config.model.model_load_path = resume_model
    if max_epoch is not None: config.training.n_epochs = max_epoch
    config.training.use_DDP = False; config.training.device = [torch.device("cuda:0")]
    set_seed(args.seed)
    runner = get_runner(config.runner, config)
    if train:
        runner.train()
    else:
        with torch.no_grad(): runner.test()
print("cwd:", os.getcwd(), "| MAX_EPOCH:", MAX_EPOCH)


## 1. 자산 확인 (VQGAN·데이터·비교 pngs)

In [ ]:
print("VQGAN ckpt:", os.path.exists("results/VQGAN/model.ckpt"))
DR = os.path.join(ROOT,"refiners/data/refine_hat")
for s in ["trainA","trainB","testA","testB"]:
    print(f"  {s}: {len(os.listdir(os.path.join(DR,s)))}장")
EVAL = os.path.join(ROOT,"runs/eval_run")
for d in ["sr_hat","cyclegan","plain_cut","unsb","hr_ref"]:
    p=os.path.join(EVAL,d); print(f"  ref {d}: {len(os.listdir(p)) if os.path.isdir(p) else 0}장")


## 2. BBDM 학습 (in-kernel, loss 스트리밍 관찰)
tqdm 진행바 + loss가 실시간으로 흐릅니다. 2 epoch마다 저장.

In [ ]:
t0=time.time()
run_bbdm(train=True, max_epoch=MAX_EPOCH)
print(f"\n학습 종료 ({time.time()-t0:.0f}s) → results/lbbdm_hat/LBBDM-f4/checkpoint/")


## 3. 샘플링 (test 60장 생성, in-kernel)
학습된 체크포인트 로드 → SB 역과정 200스텝으로 refined 생성. `sample_to_eval/200/`에 저장.

In [ ]:
cks = glob.glob("results/lbbdm_hat/LBBDM-f4/checkpoint/*.pth")
mdl = [c for c in cks if "model" in os.path.basename(c) and "optim" not in os.path.basename(c)]
mdl = sorted(mdl, key=os.path.getmtime)
assert mdl, "체크포인트 없음 — 셀2 학습 먼저"
resume = os.path.abspath(mdl[-1]); print("로드:", os.path.basename(resume))
t0=time.time()
run_bbdm(train=False, resume_model=resume)
print(f"샘플링 종료 ({time.time()-t0:.0f}s)")
SAMP = sorted(glob.glob("results/lbbdm_hat/LBBDM-f4/sample_to_eval/*"), key=os.path.getmtime)
print("샘플 경로 후보:", [os.path.basename(s) for s in SAMP])


## 4. 6-way 평가 (test 60 동일표본) — pngs만 읽음

In [ ]:
from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
def load01(p):
    im=np.asarray(Image.open(p).convert("RGB").resize((512,512))).astype(np.float32)/255.0
    return torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).float()

test_names = sorted(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
hr_ref = os.path.join(EVAL,"hr_ref")
work = os.path.join(ROOT,"runs/eval_bbdm"); hr60=os.path.join(work,"hr60"); os.makedirs(hr60,exist_ok=True)
for n in test_names:
    if not os.path.exists(os.path.join(hr60,n)):
        Image.open(os.path.join(hr_ref,n)).convert("RGB").resize((512,512)).save(os.path.join(hr60,n))
# BBDM 결과 폴더(200) → 표준이름 정리
res200 = "results/lbbdm_hat/LBBDM-f4/sample_to_eval/200"
bbdm60 = os.path.join(work,"bbdm60"); os.makedirs(bbdm60,exist_ok=True)
bmap = {os.path.splitext(os.path.basename(p))[0]:p for p in glob.glob(res200+"/*.png")}
mt=0
for n in test_names:
    src=bmap.get(os.path.splitext(n)[0]) or bmap.get(n)
    if src: Image.open(src).convert("RGB").resize((512,512)).save(os.path.join(bbdm60,n)); mt+=1
print(f"BBDM 매칭 {mt}/{len(test_names)}")

conds={"sr_hat":os.path.join(EVAL,"sr_hat"),"cyclegan":os.path.join(EVAL,"cyclegan"),
       "plain_cut":os.path.join(EVAL,"plain_cut"),"unsb":os.path.join(EVAL,"unsb"),"bbdm":bbdm60}
lp=_L.LPIPS(net="alex").to(DEV); niqe=pyiqa.create_metric("niqe",device=DEV)
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)
rows={}
for cond,dd in conds.items():
    have=[n for n in test_names if os.path.exists(os.path.join(dd,n))]
    if not have: print("skip",cond); continue
    ps,ss,lps,nqs=[],[],[],[]
    for n in have:
        out=load01(os.path.join(dd,n)); hr=load01(os.path.join(hr60,n))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1,hr.to(DEV)*2-1).mean()))
        nqs.append(float(niqe(os.path.join(dd,n))))
    tmp=os.path.join(work,f"_{cond}"); os.makedirs(tmp,exist_ok=True)
    for n in have:
        d=os.path.join(tmp,n)
        if not os.path.exists(d): os.symlink(os.path.abspath(os.path.join(dd,n)), d)
    try: fid=calculate_fid_given_paths([tmp,hr60],batch_size=50,device=DEV,dims=2048)
    except Exception as e: fid=float("nan"); print("FID err",cond,e)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print(f"\n===== 6-way (HAT 입력, test {len(test_names)}장 동일표본) =====")
print(f"{'조건':10s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*60)
for k in ["sr_hat","cyclegan","plain_cut","unsb","bbdm"]:
    if k in rows:
        p,s,l,f,n=rows[k]; print(f"{k:10s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n주의: FID test-60 소표본 → 상대비교. 육안(셀5)이 최종.")


## 5. 육안 6-way (필수)

In [ ]:
show=test_names[:3]
order=[("input","sr_hat"),("CycleGAN","cyclegan"),("plain CUT","plain_cut"),("UNSB","unsb"),("BBDM","bbdm"),("HR (GT)",None)]
fig,ax=plt.subplots(len(show),len(order),figsize=(3.1*len(order),3.1*len(show)))
for r,n in enumerate(show):
    for c,(t,key) in enumerate(order):
        p = os.path.join(hr60,n) if key is None else os.path.join(conds[key],n)
        if os.path.exists(p): ax[r,c].imshow(np.asarray(Image.open(p).convert("RGB").resize((512,512))))
        ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()
print("BBDM이 (a)선명 (b)환각/색이상 없음 (c)plain CUT보다 GT 근접 → 챔피언 교체. 아니면 plain CUT 유지.")
